<a href="https://colab.research.google.com/github/Asritha0507/ML-Market-Basket-Analysis/blob/main/03_Temporal_Training_Data_%26_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3 — Temporal and Behavioral Feature Engineering

## Objective

This notebook constructs historical, temporal, user-level, product-level, and user-product-level features for the personalized next-basket recommendation system.

The goal is to represent a customer's purchasing behavior using only information that would have been available before the customer's next basket.

The feature engineering process focuses on:

- Customer purchasing behavior
- Product popularity
- User-product affinity
- Purchase frequency
- Purchase recency
- Purchase intervals
- Basket-level behavior
- Temporal purchasing patterns

The target variable is not used while constructing historical features to avoid target leakage.

These engineered features will later be used for candidate generation, recommendation ranking, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import gc
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

base_path = "/content/drive/MyDrive/ML_Market_Basket_Analysis/Datasets"

print("Base path:", base_path)
print(os.listdir(base_path))

Mounted at /content/drive
Base path: /content/drive/MyDrive/ML_Market_Basket_Analysis/Datasets
['departments.csv', 'orders.csv', 'products.csv', 'aisles.csv', 'order_products_prior.csv', 'order_products_train.csv']


In [ ]:
orders = pd.read_csv(
    f"{base_path}/orders.csv",
    usecols=[
        "order_id",
        "user_id",
        "eval_set",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order"
    ],
    dtype={
        "order_id": "int32",
        "user_id": "int32",
        "eval_set": "category",
        "order_number": "int16",
        "order_dow": "int8",
        "order_hour_of_day": "int8",
        "days_since_prior_order": "float32"
    }
)

print("Orders shape:", orders.shape)
print("\nColumns:")
print(orders.columns.tolist())
print("\nEvaluation sets:")
print(orders["eval_set"].value_counts())

Orders shape: (3421083, 7)

Columns:
['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']

Evaluation sets:
eval_set
prior    3214874
train     131209
test       75000
Name: count, dtype: int64


### 3. Load Order-Level Historical Data

The `orders.csv` file contains information about each customer's order history.

We load only the columns required for temporal and behavioral feature engineering.

The `eval_set` column is retained because it allows us to distinguish historical orders from the target orders used later for evaluation.

The order number provides the chronological position of each order for a customer and is essential for constructing temporal features without future information.

In [ ]:
prior_orders = orders[
    orders["eval_set"] == "prior"
].copy()

train_orders = orders[
    orders["eval_set"] == "train"
].copy()

test_orders = orders[
    orders["eval_set"] == "test"
].copy()

print("Prior orders:", prior_orders.shape)
print("Train orders:", train_orders.shape)
print("Test orders:", test_orders.shape)

Prior orders: (3214874, 7)
Train orders: (131209, 7)
Test orders: (75000, 7)


### 4. Separate Historical Orders from Target Orders

The Instacart dataset provides three types of orders:

- `prior` — historical orders used to understand previous customer behavior.
- `train` — target orders for which the actual purchased products are available and can be used for model training and validation.
- `test` — future target orders whose products are hidden and will be used for final evaluation.

For temporal feature engineering, only `prior` orders are used.

This separation is essential for preventing temporal leakage. Information from a customer's target basket must never be used to calculate the features used to predict that basket.

In [ ]:
print("Maximum prior order number:",
      prior_orders["order_number"].max())

print("Minimum train order number:",
      train_orders["order_number"].min())

print("Minimum test order number:",
      test_orders["order_number"].min())

Maximum prior order number: 99
Minimum train order number: 4
Minimum test order number: 4


### 5. Verify Temporal Separation

Before creating historical features, we verify that the prior orders form the historical portion of the dataset and that the target orders occur after the available customer history.

This check helps ensure that the feature engineering pipeline follows the chronological structure of the recommendation problem.

In [ ]:
# Latest historical order for each user
last_prior_order = (
    prior_orders
    .groupby("user_id")["order_number"]
    .max()
    .reset_index(name="last_prior_order")
)

# Check train orders
train_temporal_check = train_orders.merge(
    last_prior_order,
    on="user_id",
    how="left"
)

train_temporal_check["valid_temporal_order"] = (
    train_temporal_check["order_number"]
    > train_temporal_check["last_prior_order"]
)

print(
    "Invalid train temporal rows:",
    (~train_temporal_check["valid_temporal_order"]).sum()
)

# Check test orders
test_temporal_check = test_orders.merge(
    last_prior_order,
    on="user_id",
    how="left"
)

test_temporal_check["valid_temporal_order"] = (
    test_temporal_check["order_number"]
    > test_temporal_check["last_prior_order"]
)

print(
    "Invalid test temporal rows:",
    (~test_temporal_check["valid_temporal_order"]).sum()
)

Invalid train temporal rows: 0
Invalid test temporal rows: 0


In [ ]:
order_products_prior = pd.read_csv(
    f"{base_path}/order_products_prior.csv",
    usecols=[
        "order_id",
        "product_id",
        "add_to_cart_order"
    ],
    dtype={
        "order_id": "int32",
        "product_id": "int32",
        "add_to_cart_order": "int16"
    }
)

print("Historical interactions shape:", order_products_prior.shape)
print("\nColumns:")
print(order_products_prior.columns.tolist())
print("\nSample:")
print(order_products_prior.head())

Historical interactions shape: (32434489, 3)

Columns:
['order_id', 'product_id', 'add_to_cart_order']

Sample:
   order_id  product_id  add_to_cart_order
0         2       33120                  1
1         2       28985                  2
2         2        9327                  3
3         2       45918                  4
4         2       30035                  5


### 7. Load Historical Product Interactions

The `order_products_prior.csv` file contains the products purchased in the customer's historical orders.

This is the largest file in the Instacart dataset, containing more than 32 million product-order interactions.

Only historical (`prior`) interactions are loaded because the purpose of this notebook is to construct features using information available before the target basket.

The `reordered` column is intentionally excluded from this feature-engineering dataset. It is a target-related field and must not be used when constructing historical behavioral features.

In [ ]:
prior = order_products_prior.merge(
    prior_orders[
        [
            "order_id",
            "user_id",
            "order_number",
            "order_dow",
            "order_hour_of_day",
            "days_since_prior_order"
        ]
    ],
    on="order_id",
    how="inner"
)

print("Prior interaction dataset:", prior.shape)
print("\nColumns:")
print(prior.columns.tolist())
print("\nSample:")
print(prior.head())

Prior interaction dataset: (32434489, 8)

Columns:
['order_id', 'product_id', 'add_to_cart_order', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']

Sample:
   order_id  product_id  add_to_cart_order  user_id  order_number  order_dow  \
0         2       33120                  1   202279             3          5   
1         2       28985                  2   202279             3          5   
2         2        9327                  3   202279             3          5   
3         2       45918                  4   202279             3          5   
4         2       30035                  5   202279             3          5   

   order_hour_of_day  days_since_prior_order  
0                  9                     8.0  
1                  9                     8.0  
2                  9                     8.0  
3                  9                     8.0  
4                  9                     8.0  


### 8. Combine Historical Orders with Product Interactions

The product interaction data contains product-level information, while `prior_orders` contains customer and temporal information.

The two datasets are joined using `order_id`.

The resulting dataset represents each historical product purchase together with:

- Customer identity
- Product identity
- Order sequence
- Day of week
- Order hour
- Time since the previous order
- Position of the product in the basket

This combined historical interaction table forms the foundation for user-level, product-level, and user-product-level feature engineering.

In [ ]:
user_features = (
    prior_orders
    .groupby("user_id")
    .agg(
        user_total_orders=("order_number", "max"),
        avg_days_between_orders=("days_since_prior_order", "mean")
    )
    .reset_index()
)

# Most frequently used ordering hour
favorite_hour = (
    prior_orders
    .groupby("user_id")["order_hour_of_day"]
    .agg(lambda x: x.mode().iloc[0])
    .reset_index(name="favorite_hour")
)

# Most frequently used ordering day
favorite_day = (
    prior_orders
    .groupby("user_id")["order_dow"]
    .agg(lambda x: x.mode().iloc[0])
    .reset_index(name="favorite_day")
)

# Merge user-level features
user_features = user_features.merge(
    favorite_hour,
    on="user_id",
    how="left"
)

user_features = user_features.merge(
    favorite_day,
    on="user_id",
    how="left"
)

print("User feature shape:", user_features.shape)
print("\nUser features:")
print(user_features.head())

User feature shape: (206209, 5)

User features:
   user_id  user_total_orders  avg_days_between_orders  favorite_hour  \
0        1                 10                19.555555              7   
1        2                 14                15.230769             10   
2        3                 12                12.090909             16   
3        4                  5                13.750000             11   
4        5                  4                13.333333             18   

   favorite_day  
0             1  
1             1  
2             0  
3             4  
4             3  


### 9. User-Level Behavioral Features

Customers have different purchasing habits. Some customers place many orders, while others order less frequently.

We create user-level features from historical (`prior`) orders to capture each customer's overall purchasing behavior.

The features include:

- `user_total_orders` — total number of historical orders.
- `avg_days_between_orders` — average number of days between historical orders.
- `favorite_hour` — hour at which the customer most frequently places orders.
- `favorite_day` — day of the week on which the customer most frequently places orders.

Only historical orders are used, ensuring that information from future target baskets is not included.

In [ ]:
print("Shape:", user_features.shape)

print("\nMissing values:")
print(user_features.isnull().sum())

print("\nSummary statistics:")
print(user_features.describe())

Shape: (206209, 5)

Missing values:
user_id                    0
user_total_orders          0
avg_days_between_orders    0
favorite_hour              0
favorite_day               0
dtype: int64

Summary statistics:
             user_id  user_total_orders  avg_days_between_orders  \
count  206209.000000      206209.000000            206209.000000   
mean   103105.000000          15.590367                15.209435   
std     59527.555167          16.654774                 7.105644   
min         1.000000           3.000000                 0.000000   
25%     51553.000000           5.000000                 9.416667   
50%    103105.000000           9.000000                14.500000   
75%    154657.000000          19.000000                20.285715   
max    206209.000000          99.000000                30.000000   

       favorite_hour   favorite_day  
count  206209.000000  206209.000000  
mean       12.181559       2.077121  
std         3.976492       2.060036  
min         0.000000

### 10. Validate User-Level Features

We verify the dimensions and missing values of the generated user-level features.

Since these features are calculated directly from historical orders, every customer represented in the historical dataset should have the required user-level information.

In [ ]:
product_features = (
    prior
    .groupby("product_id")
    .agg(
        product_total_orders=("order_id", "count"),
        unique_users=("user_id", "nunique")
    )
    .reset_index()
)

print("Product feature shape:", product_features.shape)

print("\nProduct features:")
print(product_features.head())

Product feature shape: (49677, 3)

Product features:
   product_id  product_total_orders  unique_users
0           1                  1852           716
1           2                    90            78
2           3                   277            74
3           4                   329           182
4           5                    15             6


### 11. Product-Level Popularity Features

Products have different levels of popularity across the entire customer population.

This section creates product-level features using the historical product interactions.

The features include:

- `product_total_orders` — total number of times a product appeared in historical baskets.
- `unique_users` — number of distinct customers who purchased the product historically.

These features capture the overall popularity and customer reach of each product.

Only historical `prior` interactions are used. The target variable is not used in calculating these features, preventing target leakage.

In [ ]:
print("Shape:", product_features.shape)

print("\nMissing values:")
print(product_features.isnull().sum())

print("\nSummary statistics:")
print(product_features.describe())

Shape: (49677, 3)

Missing values:
product_id              0
product_total_orders    0
unique_users            0
dtype: int64

Summary statistics:
         product_id  product_total_orders  unique_users
count  49677.000000          49677.000000  49677.000000
mean   24843.417356            652.907563    267.889627
std    14343.034804           4792.114416   1308.788623
min        1.000000              1.000000      1.000000
25%    12423.000000             17.000000     11.000000
50%    24842.000000             60.000000     35.000000
75%    37264.000000            260.000000    137.000000
max    49688.000000         472565.000000  73956.000000


### 12. Validate Product-Level Features

The generated product features are checked for their dimensions, missing values, and basic statistical properties.

Each product appearing in the historical interaction dataset should have corresponding popularity information.

In [ ]:
user_product_features = (
    prior
    .groupby(["user_id", "product_id"])
    .agg(
        times_purchased=("order_id", "count"),
        first_order=("order_number", "min"),
        last_order=("order_number", "max")
    )
    .reset_index()
)

user_product_features["purchase_span"] = (
    user_product_features["last_order"]
    - user_product_features["first_order"]
)

user_product_features["purchase_frequency"] = (
    user_product_features["times_purchased"]
    / (user_product_features["purchase_span"] + 1)
)

print("User-product feature shape:", user_product_features.shape)
print("\nFeatures:")
print(user_product_features.head())

User-product feature shape: (13307953, 7)

Features:
   user_id  product_id  times_purchased  first_order  last_order  \
0        1         196               10            1          10   
1        1       10258                9            2          10   
2        1       10326                1            5           5   
3        1       12427               10            1          10   
4        1       13032                3            2          10   

   purchase_span  purchase_frequency  
0              9            1.000000  
1              8            1.000000  
2              0            1.000000  
3              9            1.000000  
4              8            0.333333  


### 13. User-Product Behavioral Features

A recommendation system should not only consider whether a product is popular overall, but also how strongly a particular customer is connected to that product.

This section creates features describing the historical relationship between each user and each product.

For every user-product pair, we calculate:

- `times_purchased` — number of historical orders in which the user purchased the product.
- `first_order` — first historical order number in which the product was purchased.
- `last_order` — most recent historical order number in which the product was purchased.
- `purchase_span` — number of orders between the first and most recent purchase.
- `purchase_frequency` — purchase frequency relative to the observed purchase span.

These features represent the customer's historical affinity toward a product.

The features are calculated exclusively from `prior` interactions. The target basket and `reordered` variable are not used.

In [ ]:
print("Shape:", user_product_features.shape)

print("\nMissing values:")
print(user_product_features.isnull().sum())

print("\nDuplicate user-product pairs:")
print(
    user_product_features
    .duplicated(subset=["user_id", "product_id"])
    .sum()
)

print("\nTimes purchased statistics:")
print(user_product_features["times_purchased"].describe())

print("\nPurchase frequency statistics:")
print(user_product_features["purchase_frequency"].describe())

Shape: (13307953, 7)

Missing values:
user_id               0
product_id            0
times_purchased       0
first_order           0
last_order            0
purchase_span         0
purchase_frequency    0
dtype: int64

Duplicate user-product pairs:
0

Times purchased statistics:
count    1.330795e+07
mean     2.437226e+00
std      3.554528e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      9.900000e+01
Name: times_purchased, dtype: float64

Purchase frequency statistics:
count    1.330795e+07
mean     8.061471e-01
std      3.054965e-01
min      2.020202e-02
25%      6.000000e-01
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: purchase_frequency, dtype: float64


### 14. Validate User-Product Features

The user-product feature table is validated to ensure that:

1. Each user-product combination is represented once.
2. Purchase counts are positive for observed user-product pairs.
3. Temporal values are consistent.
4. No missing values were introduced during aggregation.
5. The features contain only historical information.

This validation is important because user-product affinity is one of the main signals used by the recommendation system.

In [ ]:
user_last_order = (
    prior_orders
    .groupby("user_id")["order_number"]
    .max()
    .reset_index(name="user_last_order")
)

user_product_features = user_product_features.merge(
    user_last_order,
    on="user_id",
    how="left"
)

user_product_features["recency"] = (
    user_product_features["user_last_order"]
    - user_product_features["last_order"]
)

print(
    user_product_features[
        [
            "user_id",
            "product_id",
            "last_order",
            "user_last_order",
            "recency"
        ]
    ].head(10)
)

print("\nRecency statistics:")
print(user_product_features["recency"].describe())

   user_id  product_id  last_order  user_last_order  recency
0        1         196          10               10        0
1        1       10258          10               10        0
2        1       10326           5               10        5
3        1       12427          10               10        0
4        1       13032          10               10        0
5        1       13176           5               10        5
6        1       14084           1               10        9
7        1       17122           5               10        5
8        1       25133          10               10        0
9        1       26088           2               10        8

Recency statistics:
count    1.330795e+07
mean     9.512767e+00
std      1.341695e+01
min      0.000000e+00
25%      1.000000e+00
50%      4.000000e+00
75%      1.200000e+01
max      9.800000e+01
Name: recency, dtype: float64


### Purchase Recency

Purchase recency measures how many orders have passed since a customer last purchased a product.

A smaller recency value indicates that the product was purchased recently, while a larger value indicates that the product has not been purchased for several orders.

Recency is useful for next-basket recommendation because recent customer-product interactions can provide a strong signal of future purchase behavior.

In [ ]:
user_product_features["avg_purchase_interval"] = (
    user_product_features["purchase_span"]
    / (user_product_features["times_purchased"] - 1)
)

user_product_features["avg_purchase_interval"] = (
    user_product_features["avg_purchase_interval"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print(
    user_product_features[
        [
            "user_id",
            "product_id",
            "times_purchased",
            "purchase_span",
            "avg_purchase_interval"
        ]
    ].head(10)
)

print("\nAverage purchase interval statistics:")
print(user_product_features["avg_purchase_interval"].describe())

   user_id  product_id  times_purchased  purchase_span  avg_purchase_interval
0        1         196               10              9                    1.0
1        1       10258                9              8                    1.0
2        1       10326                1              0                    0.0
3        1       12427               10              9                    1.0
4        1       13032                3              8                    4.0
5        1       13176                2              3                    3.0
6        1       14084                1              0                    0.0
7        1       17122                1              0                    0.0
8        1       25133                8              7                    1.0
9        1       26088                2              1                    1.0

Average purchase interval statistics:
count    1.330795e+07
mean     1.942077e+00
std      4.714519e+00
min      0.000000e+00
25%      0.0000

### Average Purchase Interval

The average purchase interval measures the average number of orders between repeated purchases of the same product by a customer.

A smaller interval indicates that the customer tends to purchase the product frequently, while a larger interval indicates less frequent purchasing behavior.

This feature complements recency by capturing the customer's historical reorder pattern rather than relying only on the most recent purchase.

In [ ]:
order_basket_sizes = (
    prior
    .groupby("order_id")
    .size()
    .reset_index(name="basket_size")
)

order_basket_sizes = order_basket_sizes.merge(
    prior_orders[["order_id", "user_id"]],
    on="order_id",
    how="left"
)

user_basket_features = (
    order_basket_sizes
    .groupby("user_id")
    .agg(
        avg_basket_size=("basket_size", "mean"),
        max_basket_size=("basket_size", "max"),
        total_items_purchased=("basket_size", "sum")
    )
    .reset_index()
)

print("Basket feature shape:", user_basket_features.shape)
print("\nBasket features:")
print(user_basket_features.head())

print("\nSummary statistics:")
print(user_basket_features.describe())

Basket feature shape: (206209, 4)

Basket features:
   user_id  avg_basket_size  max_basket_size  total_items_purchased
0        1         5.900000                9                     59
1        2        13.928571               26                    195
2        3         7.333333               11                     88
3        4         3.600000                7                     18
4        5         9.250000               12                     37

Summary statistics:
             user_id  avg_basket_size  max_basket_size  total_items_purchased
count  206209.000000    206209.000000    206209.000000          206209.000000
mean   103105.000000         9.951586        17.654865             157.289396
std     59527.555167         5.863570        10.192818             204.208233
min         1.000000         1.000000         1.000000               3.000000
25%     51553.000000         5.740741        10.000000              39.000000
50%    103105.000000         8.933333        16.000

### Historical Basket Behavior

Basket-level behavior captures the customer's typical purchasing pattern at the order level.

For each customer, we calculate:

- Average basket size
- Maximum basket size
- Total number of items purchased historically

These features provide additional context for next-basket recommendation and can help determine how many products a customer is likely to purchase in a future basket.

In [ ]:
products = pd.read_csv(
    f"{base_path}/products.csv",
    usecols=[
        "product_id",
        "aisle_id",
        "department_id"
    ],
    dtype={
        "product_id": "int32",
        "aisle_id": "int16",
        "department_id": "int16"
    }
)

print("Product metadata shape:", products.shape)
print("\nColumns:")
print(products.columns.tolist())

print("\nSample:")
print(products.head())

print("\nMissing values:")
print(products.isnull().sum())

Product metadata shape: (49688, 3)

Columns:
['product_id', 'aisle_id', 'department_id']

Sample:
   product_id  aisle_id  department_id
0           1        61             19
1           2       104             13
2           3        94              7
3           4        38              1
4           5         5             13

Missing values:
product_id       0
aisle_id         0
department_id    0
dtype: int64


### Product Metadata

Product-level behavioral features describe how popular a product is, but the product ID itself does not provide meaningful semantic information.

To provide additional product context, we incorporate aisle and department information from the product metadata.

These attributes can help the recommendation model learn similarities between products belonging to the same category or department.

In [ ]:
product_features = product_features.merge(
    products,
    on="product_id",
    how="left"
)

print("Product feature shape:", product_features.shape)

print("\nProduct features:")
print(product_features.head())

print("\nMissing values:")
print(product_features.isnull().sum())

Product feature shape: (49677, 5)

Product features:
   product_id  product_total_orders  unique_users  aisle_id  department_id
0           1                  1852           716        61             19
1           2                    90            78       104             13
2           3                   277            74        94              7
3           4                   329           182        38              1
4           5                    15             6         5             13

Missing values:
product_id              0
product_total_orders    0
unique_users            0
aisle_id                0
department_id           0
dtype: int64


### Combining Product Behavior and Product Metadata

The product feature table contains historical behavioral information such as purchase volume and number of unique customers.

The product metadata table provides structural information through aisle and department identifiers.

Combining these features gives the recommendation model both:

- Behavioral information about product popularity
- Product-category information

In [ ]:
product_features["avg_purchases_per_user"] = (
    product_features["product_total_orders"]
    / product_features["unique_users"]
)

print(
    product_features[
        [
            "product_id",
            "product_total_orders",
            "unique_users",
            "avg_purchases_per_user",
            "aisle_id",
            "department_id"
        ]
    ].head(10)
)

print("\nAverage purchases per user statistics:")
print(product_features["avg_purchases_per_user"].describe())

   product_id  product_total_orders  unique_users  avg_purchases_per_user  \
0           1                  1852           716                2.586592   
1           2                    90            78                1.153846   
2           3                   277            74                3.743243   
3           4                   329           182                1.807692   
4           5                    15             6                2.500000   
5           6                     8             5                1.600000   
6           7                    30            18                1.666667   
7           8                   165            82                2.012195   
8           9                   156            74                2.108108   
9          10                  2572          1268                2.028391   

   aisle_id  department_id  
0        61             19  
1       104             13  
2        94              7  
3        38              1  
4      

### Average Purchases per User

The number of times a product was purchased does not tell us whether it is repeatedly purchased by the same customers or broadly purchased by many different customers.

Average purchases per user captures this distinction by measuring the average number of historical purchases of a product among its unique customers.

This helps distinguish products with broad popularity from products with stronger repeat-purchase behavior.

In [ ]:
user_product_features = user_product_features.merge(
    user_features[
        [
            "user_id",
            "user_total_orders"
        ]
    ],
    on="user_id",
    how="left"
)

user_product_features["user_product_purchase_rate"] = (
    user_product_features["times_purchased"]
    / user_product_features["user_total_orders"]
)

print(
    user_product_features[
        [
            "user_id",
            "product_id",
            "times_purchased",
            "user_total_orders",
            "user_product_purchase_rate"
        ]
    ].head(10)
)

print("\nUser-product purchase rate statistics:")
print(
    user_product_features[
        "user_product_purchase_rate"
    ].describe()
)

print("\nMissing values:")
print(
    user_product_features[
        "user_product_purchase_rate"
    ].isnull().sum()
)

   user_id  product_id  times_purchased  user_total_orders  \
0        1         196               10                 10   
1        1       10258                9                 10   
2        1       10326                1                 10   
3        1       12427               10                 10   
4        1       13032                3                 10   
5        1       13176                2                 10   
6        1       14084                1                 10   
7        1       17122                1                 10   
8        1       25133                8                 10   
9        1       26088                2                 10   

   user_product_purchase_rate  
0                         1.0  
1                         0.9  
2                         0.1  
3                         1.0  
4                         0.3  
5                         0.2  
6                         0.1  
7                         0.1  
8                         0.8

### User-Product Purchase Rate

The user-product purchase rate measures how frequently a customer purchases a particular product relative to the customer's total historical orders.

A higher value indicates stronger customer-product affinity.

This feature captures personalized reorder tendency and complements the absolute number of previous purchases.

In [ ]:
print("USER FEATURES")
print("=" * 50)
print("Shape:", user_features.shape)
print("Missing values:", user_features.isnull().sum().sum())
print("Duplicate users:", user_features["user_id"].duplicated().sum())

print("\nPRODUCT FEATURES")
print("=" * 50)
print("Shape:", product_features.shape)
print("Missing values:", product_features.isnull().sum().sum())
print("Duplicate products:", product_features["product_id"].duplicated().sum())

print("\nUSER-PRODUCT FEATURES")
print("=" * 50)
print("Shape:", user_product_features.shape)
print("Missing values:", user_product_features.isnull().sum().sum())
print(
    "Duplicate user-product pairs:",
    user_product_features.duplicated(
        subset=["user_id", "product_id"]
    ).sum()
)

print("\nBASKET FEATURES")
print("=" * 50)
print("Shape:", user_basket_features.shape)
print("Missing values:", user_basket_features.isnull().sum().sum())
print("Duplicate users:", user_basket_features["user_id"].duplicated().sum())

print("\nINVALID VALUES")
print("=" * 50)

print(
    "Negative recency:",
    (user_product_features["recency"] < 0).sum()
)

print(
    "Negative purchase interval:",
    (user_product_features["avg_purchase_interval"] < 0).sum()
)

print(
    "Purchase rate outside [0,1]:",
    (
        (user_product_features["user_product_purchase_rate"] < 0) |
        (user_product_features["user_product_purchase_rate"] > 1)
    ).sum()
)

USER FEATURES
Shape: (206209, 5)
Missing values: 0
Duplicate users: 0

PRODUCT FEATURES
Shape: (49677, 6)
Missing values: 0
Duplicate products: 0

USER-PRODUCT FEATURES
Shape: (13307953, 12)
Missing values: 0
Duplicate user-product pairs: 0

BASKET FEATURES
Shape: (206209, 4)
Missing values: 0
Duplicate users: 0

INVALID VALUES
Negative recency: 0
Negative purchase interval: 0
Purchase rate outside [0,1]: 0


### Final Feature Quality Check

Before moving to candidate generation and temporal training data construction, the engineered feature tables are validated for:

- Shape
- Missing values
- Duplicate keys
- Invalid numerical values
- Feature ranges

This ensures that the historical feature tables are consistent before they are used for next-basket recommendation.

In [ ]:
feature_path = f"{base_path}/Features"

os.makedirs(feature_path, exist_ok=True)

user_features.to_parquet(
    f"{feature_path}/user_features.parquet",
    index=False
)

product_features.to_parquet(
    f"{feature_path}/product_features.parquet",
    index=False
)

user_basket_features.to_parquet(
    f"{feature_path}/user_basket_features.parquet",
    index=False
)

user_product_features.to_parquet(
    f"{feature_path}/user_product_features.parquet",
    index=False
)

print("Feature tables saved successfully.")
print("\nSaved files:")
print(os.listdir(feature_path))

Feature tables saved successfully.

Saved files:
['user_features.parquet', 'product_features.parquet', 'user_basket_features.parquet', 'user_product_features.parquet']


### Saving Engineered Feature Tables

The engineered feature tables are saved as intermediate datasets.

Parquet format is used for the large user-product feature table because it provides efficient storage and faster loading compared with CSV while preserving numerical data types.

These saved tables will be reused during candidate generation and temporal training-data construction.

In [ ]:
print("NOTEBOOK 3 FINAL CHECK")
print("=" * 50)

print("User features:", user_features.shape)
print("Product features:", product_features.shape)
print("Basket features:", user_basket_features.shape)
print("User-product features:", user_product_features.shape)

print("\nFeature files available:")
for file in os.listdir(feature_path):
    print("-", file)

print("\nNotebook 3 completed successfully.")
print("Ready for Notebook 4: Temporal Candidate Generation and Training Dataset Construction.")

NOTEBOOK 3 FINAL CHECK
User features: (206209, 5)
Product features: (49677, 6)
Basket features: (206209, 4)
User-product features: (13307953, 12)

Feature files available:
- user_features.parquet
- product_features.parquet
- user_basket_features.parquet
- user_product_features.parquet

Notebook 3 completed successfully.
Ready for Notebook 4: Temporal Candidate Generation and Training Dataset Construction.
